# Coordination and Delegation in Multi-Agent Systems

The orchestrator-worker pattern from NB08 uses raw LLM calls as workers. The next step is making each worker a *full agent* — its own context window, its own tools, its own reasoning loop. This notebook explores multi-agent systems: why single agents hit a ceiling on complex tasks, how the CDA library's `SubAgentTool` handles sub-agent delegation, how to chain agents into sequential pipelines, and how to implement LLM-driven speaker selection for collaborative discussions.

The guiding question throughout is not "how do we make the system more complicated?" but rather "when does the added complexity actually pay off?" Multi-agent architectures are more expensive, harder to debug, and more likely to fail in unexpected ways than single-agent systems. The goal is to understand exactly which problems justify them.

## Single Agent Limitations

A single agent loop handles most tasks well, but it runs into three structural problems on complex, multi-step work:

**Context pollution.** Every tool call appends its output to the conversation. A task that involves investigation, then fixing, then writing tests, then updating documentation fills the context with accumulated output from each phase. By the time the agent reaches documentation, it is reasoning over a window bloated with code diffs, grep results, and test output — signal from earlier phases degrades into noise.

**Role confusion.** A single system prompt cannot simultaneously optimize for careful code review (skeptical, detailed, line-by-line) and creative test writing (generative, coverage-oriented, adversarial). The agent ends up as a generalist when the task demands specialists.

**Skill concentration.** Some tasks require genuinely different competencies: reading and classifying code vs. writing prose vs. auditing security. A single system prompt averages these out.

Multi-agent architectures address all three: each sub-agent starts with a clean context window, carries a role-specific system prompt, and works on a well-scoped subtask. The parent agent coordinates but does not execute.

:::{.callout-note}
The cost of these benefits is real: more LLM calls, higher total token usage, harder debugging, and more failure modes. Single-agent is the right default. Move to multi-agent only when you have evidence that context pollution or role confusion is actually degrading quality.

:::

## Setup

**Setup.** Imports and CDA session initialization:

In [ ]:
import os
import asyncio
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI

from notebooks.agent.config import Config, ApprovalPolicy
from notebooks.agent.session import Session
from notebooks.agent.agent import Agent
from notebooks.agent.events import AgentEventType
from notebooks.agent.tools.subagents import SubAgentTool, BUILTIN_ROLES, SubAgentParams

load_dotenv()

We set up a helper to run an agent and collect its final response and token usage:

In [ ]:
async def run_agent(task: str, config: Config) -> tuple[str, dict]:
    """Run a CDA Agent on *task* and return (response, usage)."""
    session = Session(config)
    agent = Agent(config=config, session=session)
    response = ""
    usage = {}
    async for event in agent.run(task):
        if event.type == AgentEventType.AGENT_END:
            response = event.data.get("response", "") or ""
            usage = event.data.get("usage") or {}
    return response, usage

## The SubAgentTool

The CDA library implements sub-agent delegation through `SubAgentTool` in `tools/subagents.py`. This tool allows a parent agent to spawn a focused child agent as a regular tool call — the parent calls `run_sub_agent(task=..., role=...)`, the child runs to completion, and the child's final response is returned as the tool result.

The child agent is fully isolated:
- Starts with a fresh `Session` (empty conversation history)
- Uses a role-specific system prompt from `BUILTIN_ROLES`
- Runs with `ApprovalPolicy.YOLO` so it operates unattended
- Can have its tool set restricted to match the role (e.g., read-only tools for an investigator)

The five built-in roles are:

In [ ]:
for role, (prompt, tools) in BUILTIN_ROLES.items():
    tool_str = ", ".join(tools) if tools else "all tools"
    print(f"  {role}")
    print(f"    tools: {tool_str}")
    print(f"    prompt: {prompt[:80]}...")
    print()

### How the tool works

When the parent agent calls `run_sub_agent`, `SubAgentTool.execute()` runs the following steps:

1. Resolve the role to a `(system_prompt, allowed_tools)` pair from `BUILTIN_ROLES` (or the caller-supplied `system_prompt` for `role="custom"`).
2. Build a child `Config` that inherits the parent's model and working directory but overrides `approval=YOLO`, `max_turns`, `allowed_tools`, and `developer_instructions`.
3. Create a fresh `Session(child_config)` and replace its system message with the role prompt.
4. Create a `Agent(config, session)` and stream `agent.run(task)`.
5. Collect the child's final `TEXT_COMPLETE` event and return it as a `ToolResult`.

The result is prefixed with `[Sub-agent result | role=<name>]` so the parent can clearly see where the output came from.

:::{.callout-note}
`SubAgentTool` is registered in the default tool registry only when `Config.allowed_tools` includes `"run_sub_agent"` or is `None` (all tools allowed). This means it is available in a standard `Session` by default.

:::

## Sub-Agent Delegation

We run a parent agent on a task that naturally decomposes into investigation followed by documentation. The parent is free to use `run_sub_agent` to delegate the investigation to a specialized child.

**Config.** The parent runs with `ON_REQUEST` approval so sub-agent calls are visible:

In [ ]:
parent_config = Config(
    approval=ApprovalPolicy.ON_REQUEST,
    max_turns=10,
)

**Delegation task.** We ask the parent to investigate the CDA library structure and summarize it. Investigation is well-scoped, read-only work — exactly what the `codebase_investigator` role is designed for:

In [ ]:
task = (
    "Use a sub-agent to investigate the CDA library at src/notebooks/agent/ "
    "and report: (1) which modules it contains, (2) which classes are exported, "
    "and (3) how many builtin tools are registered by default. "
    "Summarize the findings in a short paragraph."
)

response, usage = await run_agent(task, parent_config)
print(response)

The annotations below describe the key steps:

1. The parent agent decides to call `run_sub_agent` with `role="codebase_investigator"` — it does not call any file tools directly.
2. `SubAgentTool.execute()` creates a child agent with restricted tools (`read_file`, `list_dir`, `glob`, `grep`, `shell`) and runs it.
3. The child uses those tools to explore `src/notebooks/agent/`, collects its findings, and produces a structured report.
4. The child's final text is returned to the parent as the tool result.
5. The parent uses that report to write the summary paragraph.

### Custom roles

Built-in roles cover the most common patterns, but `role="custom"` combined with a `system_prompt` and `allowed_tools` handles any specialization. We demonstrate with a security auditor role that checks for hardcoded secrets — a common and genuinely useful sub-agent pattern.

In [ ]:
custom_task = (
    "Use a sub-agent with role='custom' to scan the src/notebooks/agent/ directory "
    "for any hardcoded secrets, API keys, or passwords. The sub-agent should only "
    "be allowed to use read_file, glob, and grep. Report what it found."
)

custom_system = (
    "You are a security auditor. Your job is to scan source code for hardcoded "
    "secrets: API keys, passwords, tokens, private keys, or any string that looks "
    "like a credential. Search thoroughly. Report each finding with its file path "
    "and line number. If nothing is found, explicitly state that. Do NOT modify files."
)

# Build a config that includes the custom system prompt in the task description
# so the parent agent passes it through to the sub-agent.
audit_task = (
    f"Use run_sub_agent with role='custom', "
    f"system_prompt='{custom_system}', "
    "allowed_tools=['read_file', 'glob', 'grep'], "
    f"and task: 'Scan src/notebooks/agent/ for hardcoded secrets, API keys, or passwords.'"
)

audit_config = Config(approval=ApprovalPolicy.YOLO, max_turns=8)
response, _ = await run_agent(audit_task, audit_config)
print(response)

## Sequential Pipeline

Sub-agent delegation keeps the parent in the loop — it orchestrates, tools execute. For tasks with a fixed, well-ordered sequence of stages, a **sequential pipeline** is simpler: we run sub-agents explicitly in Python, passing each stage's output directly to the next.

The advantage over single-agent delegation is the same — clean context, specialized roles — but the pipeline is deterministic and transparent: the stages, the order, and the data flow are all visible in Python code rather than inferred from LLM reasoning.

### Pipeline implementation

We build a two-stage code improvement pipeline: a reviewer identifies issues in a file, then a fixer applies targeted corrections based on the review. Each stage is a standalone sub-agent called from Python rather than from another agent.

In [ ]:
async def run_sub_agent_directly(
    task: str,
    role: str,
    system_prompt: str | None = None,
    allowed_tools: list[str] | None = None,
    max_turns: int = 20,
) -> tuple[str, dict]:
    """Run a sub-agent directly (not via a parent agent) and return (response, usage)."""
    from notebooks.agent.tools.subagents import BUILTIN_ROLES
    from notebooks.agent.tools.registry import create_default_registry

    if role == "custom":
        if not system_prompt:
            raise ValueError("role='custom' requires a system_prompt")
        role_prompt = system_prompt
        role_tools = allowed_tools
    else:
        role_prompt, role_tools = BUILTIN_ROLES[role]
        if allowed_tools is not None:
            role_tools = allowed_tools  # caller override

    child_config = Config(
        approval=ApprovalPolicy.YOLO,
        max_turns=max_turns,
        allowed_tools=role_tools,
        developer_instructions=role_prompt,
    )
    child_session = Session(child_config)
    child_session.messages[0]["content"] = role_prompt

    child_agent = Agent(config=child_config, session=child_session)
    response = ""
    usage = {}
    async for event in child_agent.run(task):
        if event.type == AgentEventType.AGENT_END:
            response = event.data.get("response", "") or ""
            usage = event.data.get("usage") or {}
    return response, usage

**Stage 1 — Review.** The `code_reviewer` role scans the target file and produces a structured report:

In [ ]:
TARGET_FILE = "src/notebooks/agent/loop_detector.py"

review_task = (
    f"Review {TARGET_FILE} for correctness, style, and performance issues. "
    "Produce a structured list of issues with file path and line number for each. "
    "Focus on actionable problems, not style nits."
)

print("Stage 1: reviewing...")
review_report, review_usage = await run_sub_agent_directly(
    task=review_task,
    role="code_reviewer",
    max_turns=15,
)
print(f"Review tokens: {review_usage.get('total_tokens', 'N/A')}")
print()
print(review_report)

**Stage 2 — Fix.** The `code_fixer` role receives the review report as context and applies targeted fixes:

In [ ]:
fix_task = (
    f"The following review was produced for {TARGET_FILE}:\n\n"
    f"{review_report}\n\n"
    "Apply the most important fix from this review. Make a minimal, targeted change. "
    "Report exactly what you changed and why, including the line numbers affected."
)

print("Stage 2: fixing...")
fix_report, fix_usage = await run_sub_agent_directly(
    task=fix_task,
    role="code_fixer",
    max_turns=15,
)
print(f"Fix tokens: {fix_usage.get('total_tokens', 'N/A')}")
print()
print(fix_report)

The annotations below describe the pipeline data flow:

1. Stage 1 runs with restricted tools (read-only) and produces a structured review.
2. The review report is embedded verbatim into the Stage 2 task description — this is the stage-to-stage handoff.
3. Stage 2 receives the full context it needs in its *initial user message* rather than accumulating it through tool calls — its context window starts clean.
4. Each stage's `usage` is tracked independently, so we can see the per-stage cost.

### Token cost comparison

Pipelines use more total tokens than a single agent doing the same task — each stage pays the overhead of the system prompt and role preamble. The tradeoff is cleaner reasoning per stage:

In [ ]:
total_pipeline_tokens = (
    review_usage.get("total_tokens", 0) + fix_usage.get("total_tokens", 0)
)
print(f"Pipeline total tokens: {total_pipeline_tokens}")
print(f"  Stage 1 (review):  {review_usage.get('total_tokens', 0)}")
print(f"  Stage 2 (fix):     {fix_usage.get('total_tokens', 0)}")

## LLM-Driven Speaker Selection

Pipelines fix the sequence of speakers in advance. An alternative is to let an LLM *choose* which agent should speak next based on the conversation so far. This is the **speaker selection** pattern — a lightweight form of the approach used in AutoGen and similar frameworks.

The setup has three components:

- **Agents:** Two or more agents with distinct specializations and system prompts
- **Shared history:** A list of `{role, speaker, content}` messages visible to all agents and the moderator
- **Moderator:** A lightweight LLM call that reads the history and outputs the name of the next speaker, or `"DONE"` to terminate

The conversation runs until the moderator says `DONE` or a maximum turn limit is reached.

**Agents.** We define two agents with different perspectives on software design — a Python expert who favors simplicity and a performance expert who focuses on efficiency. Both use raw `AsyncOpenAI` calls since they are single-turn responders, not full agentic loops:

In [ ]:
oai = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
MODEL = "anthropic/claude-sonnet-4"

AGENTS = {
    "Python Expert": (
        "You are a Python expert. You favor clear, idiomatic Python code and "
        "the standard library. You value simplicity and readability. "
        "Keep your responses focused and concise — 2-4 sentences."
    ),
    "Performance Expert": (
        "You are a performance engineer. You focus on algorithmic efficiency, "
        "memory usage, and throughput. You challenge solutions that sacrifice "
        "performance for convenience. "
        "Keep your responses focused and concise — 2-4 sentences."
    ),
}

**Moderator.** The moderator sees the full conversation and returns either a speaker name or `DONE`:

In [ ]:
async def select_next_speaker(
    history: list[dict],
    agents: dict[str, str],
    topic: str,
) -> str:
    """Return the name of the next speaker, or 'DONE'."""
    agent_names = list(agents.keys())
    history_text = "\n".join(
        f"[{msg['speaker']}]: {msg['content']}" for msg in history
    )
    prompt = (
        f"You are moderating a discussion on: '{topic}'\n\n"
        f"Participants: {', '.join(agent_names)}\n\n"
        f"Conversation so far:\n{history_text}\n\n"
        f"Choose who should speak next to advance the discussion, or respond "
        f"'DONE' if the discussion has reached a useful conclusion.\n"
        f"Respond with ONLY one of: {', '.join(agent_names)}, or DONE."
    )
    response = await oai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=20,
    )
    choice = response.choices[0].message.content.strip()
    # Normalize: accept partial matches
    for name in agent_names:
        if name.lower() in choice.lower():
            return name
    return "DONE"

**Agent response.** Each agent receives the full conversation as context and produces a single response:

In [ ]:
async def agent_respond(
    speaker: str,
    agents: dict[str, str],
    history: list[dict],
    topic: str,
) -> str:
    """Generate a response from *speaker* given the conversation history."""
    system = agents[speaker]
    history_text = "\n".join(
        f"[{msg['speaker']}]: {msg['content']}" for msg in history
    ) or "(no messages yet)"
    user = (
        f"Topic: {topic}\n\n"
        f"Conversation so far:\n{history_text}\n\n"
        "Please contribute your perspective."
    )
    response = await oai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()

**Discussion loop.** The loop alternates between the moderator selecting a speaker and the selected agent responding:

In [ ]:
async def run_discussion(
    topic: str,
    agents: dict[str, str],
    max_turns: int = 6,
) -> list[dict]:
    """Run a moderated multi-agent discussion and return the full history."""
    history: list[dict] = []
    for turn in range(max_turns):
        speaker = await select_next_speaker(history, agents, topic)
        if speaker == "DONE":
            print(f"Moderator: DONE (turn {turn + 1})")
            break
        content = await agent_respond(speaker, agents, history, topic)
        history.append({"speaker": speaker, "content": content})
        print(f"[{speaker}]: {content[:120]}..." if len(content) > 120 else f"[{speaker}]: {content}")
        print()
    return history

**Run.** We pose a design question that benefits from multiple perspectives — caching layer design has both correctness and performance dimensions:

In [ ]:
topic = "How should we implement a caching layer for a high-read Python web API?"

history = await run_discussion(topic, AGENTS, max_turns=6)
print(f"Discussion concluded after {len(history)} turns.")

### When to use speaker selection

| Use it when | Avoid it when |
|-------------|---------------|
| Task benefits from multiple viewpoints | Task has a clear, fixed sequence |
| Optimal speaker order is not known in advance | Moderator overhead is not justified |
| Agents have complementary, not overlapping, expertise | Simple tasks where one agent suffices |
| Emergent discussion is more useful than structured pipeline | You need deterministic, auditable behavior |

:::{.callout-caution}
Speaker selection adds two LLM calls per turn: one for the moderator and one for the agent. On long discussions this cost compounds quickly. Prefer pipelines when the stage order is fixed.

:::

## Single vs. Multi-Agent

We compare two approaches on the same code review task: a single agent doing everything in one loop vs. a two-stage pipeline. The comparison measures total tokens (cost proxy) and subjective output quality.

**Single agent.** One agent performs investigation, review, and summary in a single continuous loop:

In [ ]:
REVIEW_TARGET = "src/notebooks/agent/loop_detector.py"

single_task = (
    f"Thoroughly review {REVIEW_TARGET}: read it, identify all issues, "
    "and produce a structured report with file path and line number for each issue."
)

single_config = Config(approval=ApprovalPolicy.YOLO, max_turns=20)
single_response, single_usage = await run_agent(single_task, single_config)

print(f"Single agent tokens: {single_usage.get('total_tokens', 'N/A')}")
print()
print(single_response[:600])

**Pipeline.** Two specialized agents — investigator then reviewer — each with a clean context:

In [ ]:
# Stage 1: investigate structure and purpose
invest_task = (
    f"Read {REVIEW_TARGET} and produce a concise summary of: "
    "(1) what the module does, (2) its public API, and (3) any data structures used."
)
invest_report, invest_usage = await run_sub_agent_directly(
    task=invest_task,
    role="codebase_investigator",
    max_turns=10,
)

# Stage 2: review with investigation context already provided
pipeline_review_task = (
    f"Module summary:\n{invest_report}\n\n"
    f"Now review the full source of {REVIEW_TARGET} for correctness, style, and "
    "performance. Produce a structured report with file path and line number."
)
pipeline_review, pipeline_usage = await run_sub_agent_directly(
    task=pipeline_review_task,
    role="code_reviewer",
    max_turns=15,
)

total_pipeline = invest_usage.get("total_tokens", 0) + pipeline_usage.get("total_tokens", 0)
print(f"Pipeline total tokens: {total_pipeline}")
print(f"  Stage 1 (investigate): {invest_usage.get('total_tokens', 0)}")
print(f"  Stage 2 (review):      {pipeline_usage.get('total_tokens', 0)}")
print()
print(pipeline_review[:600])

**Comparison.**

In [ ]:
print("Approach              | Total tokens | Notes")
print("-" * 60)
print(f"Single agent          | {single_usage.get('total_tokens', 0):>12} | One loop, all roles")
print(f"Two-stage pipeline    | {total_pipeline:>12} | Investigator + reviewer")

### Decision guidelines

The right architecture depends on the task, not on maximizing sophistication:

| Scenario | Recommendation |
|----------|----------------|
| Well-scoped task, fits in context | Single agent |
| Task has distinct sequential stages | Sequential pipeline |
| Subtasks are unknown until the parent reasons | Sub-agent delegation |
| Task benefits from multiple viewpoints | Speaker selection |
| Simple task, cost-sensitive | Single agent |

:::{.callout-important}
Multi-agent systems are harder to debug than single-agent ones. When a pipeline fails, you need to identify which stage failed and why — which requires logging each stage's input and output. Always log the full output of each sub-agent during development.

:::

---

■